# Customer Retention Intelligence — Iteration 2: Explainability

This notebook adds the minimum explainability layer needed for the final assistant.

By the end of this notebook, you will be able to:

- calculate churn probability for a customer
- explain why one customer is likely to churn
- see which factors generally drive churn across the dataset
- prepare the explanation outputs for the later chat interface

This notebook expects:

- dataset: `../data/Customer_Churn_Dataset.xlsx`
- trained model: `../models/baseline_retention_model.joblib`


## 0. Install SHAP if needed

If the import below fails, run this once in your terminal:

```bash
pip install shap
```

Also add `shap` to your `requirements.txt`.


In [ ]:
import pandas as pd
import numpy as np
import shap
import joblib
import matplotlib.pyplot as plt


## 1. Load the trained model

In [ ]:
model = joblib.load("../models/baseline_retention_model.joblib")

print("Model loaded successfully.")


## 2. Load the dataset

In [ ]:
df = pd.read_excel(
    "../data/Customer_Churn_Dataset.xlsx",
    sheet_name="01 Churn-Dataset"
)

print("Dataset shape:", df.shape)
df.head()


## 3. Prepare the data exactly like Iteration 1

In [ ]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

customer_ids = df["customerID"].copy()

df_model = df.drop(columns=["customerID"]).copy()

df_model["Churn"] = df_model["Churn"].replace({
    "No": 0,
    "Yes": 1
})

X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts())


## 4. Extract the preprocessing step and classifier

The saved model is a scikit-learn `Pipeline`.

We separate:
- the preprocessing logic
- the Logistic Regression classifier


In [ ]:
preprocessor = model.named_steps["preprocessor"]
classifier = model.named_steps["classifier"]

print("Preprocessor:", type(preprocessor).__name__)
print("Classifier:", type(classifier).__name__)


## 5. Transform the customer features

In [ ]:
X_processed = preprocessor.transform(X)

feature_names = preprocessor.get_feature_names_out()

print("Processed shape:", X_processed.shape)
print("Number of processed features:", len(feature_names))
print()
print("First 20 processed feature names:")
print(feature_names[:20])


## 6. Create the SHAP explainer

Because the baseline model is Logistic Regression, we use `LinearExplainer`.


In [ ]:
explainer = shap.LinearExplainer(
    classifier,
    X_processed
)

shap_values = explainer(X_processed)

print("SHAP values created successfully.")
print("SHAP shape:", shap_values.values.shape)


## 7. General churn drivers

This plot helps answer:

> Why are customers generally churning?

Features near the top have the strongest overall effect on the model's predictions.


In [ ]:
shap.plots.beeswarm(
    shap_values,
    max_display=15
)


## 8. Rank the overall most important churn drivers

This gives us a table we can later use in the assistant.


In [ ]:
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

global_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values(
    by="mean_abs_shap",
    ascending=False
).reset_index(drop=True)

global_importance.head(20)


## 9. Predict one customer's churn risk

Change `customer_index` to inspect a different customer.


In [ ]:
customer_index = 0

customer = X.iloc[[customer_index]]
customer_id = customer_ids.iloc[customer_index]

probability = model.predict_proba(customer)[0][1]
prediction = model.predict(customer)[0]

risk_level = (
    "High" if probability >= 0.60
    else "Medium" if probability >= 0.30
    else "Low"
)

print("Customer ID:", customer_id)
print("Predicted churn:", prediction)
print("Churn probability:", round(probability, 3))
print("Risk level:", risk_level)


## 10. Visual explanation for one customer

This waterfall plot shows which features pushed the prediction toward churn or toward staying.


In [ ]:
shap.plots.waterfall(
    shap_values[customer_index],
    max_display=12
)


## 11. Convert one customer's SHAP values into a table

This is important because the later chatbot should use structured evidence, not guess.


In [ ]:
customer_shap = pd.DataFrame({
    "feature": feature_names,
    "shap_value": shap_values.values[customer_index]
})

customer_shap["absolute_impact"] = customer_shap["shap_value"].abs()

customer_shap = customer_shap.sort_values(
    by="absolute_impact",
    ascending=False
).reset_index(drop=True)

customer_shap.head(15)


## 12. Separate churn-increasing and churn-reducing factors

Positive SHAP values push the model more toward churn.

Negative SHAP values push the model more toward staying.


In [ ]:
risk_factors = customer_shap[
    customer_shap["shap_value"] > 0
].head(5)

protective_factors = customer_shap[
    customer_shap["shap_value"] < 0
].head(5)

print("Top factors increasing churn risk:")
display(risk_factors[["feature", "shap_value"]])

print("\nTop factors reducing churn risk:")
display(protective_factors[["feature", "shap_value"]])


## 13. Create a reusable customer explanation function

This function will later be called by the chat interface.


In [ ]:
def explain_customer(customer_id, top_n=5):
    matches = customer_ids[customer_ids == customer_id].index

    if len(matches) == 0:
        return {
            "error": f"Customer {customer_id} was not found."
        }

    idx = matches[0]

    customer_row = X.iloc[[idx]]
    probability = model.predict_proba(customer_row)[0][1]
    prediction = model.predict(customer_row)[0]

    if probability >= 0.60:
        risk_level = "High"
    elif probability >= 0.30:
        risk_level = "Medium"
    else:
        risk_level = "Low"

    shap_df = pd.DataFrame({
        "feature": feature_names,
        "shap_value": shap_values.values[idx]
    })

    shap_df["absolute_impact"] = shap_df["shap_value"].abs()
    shap_df = shap_df.sort_values(
        by="absolute_impact",
        ascending=False
    )

    risk = shap_df[
        shap_df["shap_value"] > 0
    ].head(top_n)

    protective = shap_df[
        shap_df["shap_value"] < 0
    ].head(top_n)

    return {
        "customer_id": customer_id,
        "churn_probability": float(probability),
        "predicted_churn": int(prediction),
        "risk_level": risk_level,
        "risk_factors": risk[["feature", "shap_value"]].to_dict("records"),
        "protective_factors": protective[["feature", "shap_value"]].to_dict("records")
    }


## 14. Test the function

In [ ]:
test_customer_id = customer_ids.iloc[0]

explanation = explain_customer(test_customer_id)

explanation


## 15. Create a reusable overall churn explanation function

This will support questions such as:

> Why are customers generally churning?


In [ ]:
def explain_overall_churn(top_n=10):
    top_drivers = global_importance.head(top_n)

    return {
        "overall_churn_rate": float(y.mean()),
        "top_drivers": top_drivers.to_dict("records")
    }


## 16. Test the overall explanation

In [ ]:
overall_explanation = explain_overall_churn()

overall_explanation


# Iteration 2 complete

You now have the core evidence layer needed for your final assistant.

You can now:

- predict churn probability for a customer
- explain why that customer is at risk
- identify factors reducing that customer's risk
- identify the strongest overall churn drivers
- return explanations as structured Python dictionaries

## Next iteration

We will add only what is necessary for your final goal:

1. translate technical feature names into readable business language
2. create retention recommendations from actionable risk factors
3. build a simple natural-language question interface

After that, you will be able to ask questions like:

- "Why is this customer likely to churn?"
- "What can make this customer stay?"
- "Why are customers generally churning?"
